In [8]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load pre-trained model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="multi_label_classification")

def preprocess_data(texts, labels, max_length=512):
    input_ids = []
    attention_masks = []
    for text in texts:
        encoded = tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids.append(encoded['input_ids'])
        attention_masks.append(encoded['attention_mask'])

    return torch.cat(input_ids, dim=0), torch.cat(attention_masks, dim=0), torch.tensor(labels)

# Example data
texts = ["This is a positive review.", "This is a negative review."]
labels = [[1, 0], [0, 1]]  # Multi-label format
input_ids, attention_masks, labels = preprocess_data(texts, labels)

from torch.utils.data import DataLoader, TensorDataset, random_split

dataset = TensorDataset(input_ids, attention_masks, labels)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

from transformers import AdamW, get_linear_schedule_with_warmup

num_epochs = 1

optimizer = AdamW(model.parameters(), lr=5e-5)
total_steps = len(train_dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# Training loop
for epoch in range(num_epochs):
    model.train()
    for batch in train_dataloader:
        b_input_ids, b_attention_mask, b_labels = batch
        outputs = model(b_input_ids, attention_mask=b_attention_mask, labels=b_labels.float())
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    model.eval()
    for batch in val_dataloader:
        with torch.no_grad():
            b_input_ids, b_attention_mask, b_labels = batch
            outputs = model(b_input_ids, attention_mask=b_attention_mask, labels=b_labels.float())
            val_loss = outputs.loss
            # Optionally, compute evaluation metrics

print("Training complete!")

C:\Users\RAYMOND\AppData\Local\pypoetry\Cache\virtualenvs\text2uml-yWP-Bm0a-py3.9\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you 

Training complete!


In [ ]:
glxinfo 

In [15]:
# Function to preprocess a single text for inference
def preprocess_single_text(text, tokenizer, max_length=512):
    encoded = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    return encoded['input_ids'], encoded['attention_mask']

# Example text for inference
text = "This is a positive revieWing."

# Preprocess the text
input_ids, attention_mask = preprocess_single_text(text, tokenizer)

# Put the model in evaluation mode
model.eval()

# Perform inference
with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits

# Convert logits to probabilities (optional)
probabilities = torch.sigmoid(logits)

# Interpret the probabilities to get the predicted labels
threshold = 0.5
predicted_labels = (probabilities > threshold).int()

# Print the logits, probabilities, and predicted labels
print("Logits:", logits)
print("Probabilities:", probabilities)
print("Predicted Labels:", predicted_labels)

Logits: tensor([[0.2181, 1.2243]])
Probabilities: tensor([[0.5543, 0.7728]])
Predicted Labels: tensor([[1, 1]], dtype=torch.int32)


# Try Spacy

In [ ]:
import spacy

# Load the spaCy model
nlp = spacy.load("en_core_web_sm")

# Function to lemmatize text
def lemmatize_text(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc]

# Example usage
text = "i am drowning, under the sea, but probably tired of this tho!"
lemmas = lemmatize_text(text)
print(lemmas)

In [ ]:
import spacy
from transformers import BertTokenizer
# Load the spaCy model
nlp = spacy.load("en_core_web_sm")

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Function to tokenize and lemmatize text
def tokenize_and_lemmatize(text):
    # Tokenize the text using BERT tokenizer
    tokens = tokenizer.tokenize(text)
    
    # Convert tokens back to a string
    tokenized_text = " ".join(tokens)
    
    # Lemmatize the tokenized text using spaCy
    doc = nlp(tokenized_text)
    lemmas = [token.lemma_ for token in doc]
    
    return lemmas

# Example usage
text = "i am drowning, under the sea, but probably tired of this tho!"

lemmas = tokenize_and_lemmatize(text)
print(lemmas)

In [1]:
import numpy as np

# Example text and label embeddings (from BERT, for instance)
# Assuming embeddings are numpy arrays of shape (num_texts, embedding_dim) and (num_labels, embedding_dim)
text_embeddings = np.random.rand(5, 768)   # e.g., 5 text embeddings of dimension 768
label_embeddings = np.random.rand(3, 768)  # e.g., 3 label embeddings of dimension 768

# Initialize matrix G with shape (num_texts, num_labels)
G = np.zeros((text_embeddings.shape[0], label_embeddings.shape[0]))

# Calculate matrix G
for i, text_vector in enumerate(text_embeddings):
    for j, label_vector in enumerate(label_embeddings):
        # Dot product between text and label vectors
        dot_product = np.dot(text_vector, label_vector)
        
        # Norms of the text and label vectors
        norm_text = np.linalg.norm(text_vector)
        norm_label = np.linalg.norm(label_vector)
        
        # Calculate correlation and store in G
        G[i, j] = dot_product / (norm_text * norm_label)

print("Matrix G:")
print(G)


Matrix G:
[[0.7576772  0.75828001 0.76050249]
 [0.74109526 0.75500508 0.74695557]
 [0.75112674 0.74067202 0.7355393 ]
 [0.75786368 0.75710317 0.73343214]
 [0.75890876 0.74246641 0.75565786]]


In [3]:
import torch
from transformers import BertTokenizer, BertModel

# Load pre-trained BERT model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

# Function to get embeddings for a given text
def get_embeddings(text):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    
    # Get the embeddings from the BERT model
    with torch.no_grad():
        outputs = model(**inputs)
    
    # The embeddings are in the last hidden state
    embeddings = outputs.last_hidden_state
    return embeddings

# Example usage
text = "1 0"
embeddings = get_embeddings(text)
print(embeddings)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tensor([[[-0.6825, -0.1290,  0.1570,  ...,  0.1064,  0.5641,  0.7305],
         [-1.1260, -0.0131, -0.1717,  ..., -0.1907,  0.8323,  0.7413],
         [-1.4837, -0.5426,  0.2849,  ...,  0.7560,  0.2310,  0.2384],
         [ 0.9823,  0.1584, -0.2460,  ...,  0.3010, -0.6670, -0.3094]]])


In [1]:
import torch
from transformers import BertTokenizer, BertForTokenClassification, Trainer, TrainingArguments
from transformers import pipeline
from sklearn.model_selection import train_test_split

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Define IBO tags
IBO_tags = {"O": 0, "B-CLASS": 1, "I-CLASS": 2, "B-ATTR": 3, "I-ATTR": 4}

# Initialize BERT tokenizer and model for token classification
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForTokenClassification.from_pretrained(model_name, num_labels=len(IBO_tags))
model.to(device)

# Sample training data (tokens and IBO tags)
training_data = [
    {"tokens": ["John", "Doe", "is", "a", "student", "in", "Computer", "Science"], 
     "labels": ["B-CLASS", "I-CLASS", "O", "O", "O", "O", "B-ATTR", "I-ATTR"]},
    {"tokens": ["Mary", "Smith", "works", "at", "a", "software", "company"], 
     "labels": ["B-CLASS", "I-CLASS", "O", "O", "O", "B-ATTR", "I-ATTR"]},
]

# Convert the tokens and labels into BERT-compatible input format
def encode_examples(data, tokenizer):
    encodings = tokenizer(data["tokens"], is_split_into_words=True, return_offsets_mapping=True, padding='max_length', truncation=True, max_length=16)
    labels = [IBO_tags[label] for label in data["labels"]]
    # Align labels to tokenized output
    label_ids = [-100] * len(encodings["input_ids"])
    for idx, label in enumerate(labels):
        label_ids[idx + 1] = label  # Skip [CLS] token
    encodings["labels"] = label_ids
    return encodings

# Preprocess training and test data
train_encodings = [encode_examples(item, tokenizer) for item in training_data]
train_dataset = torch.utils.data.Dataset.from_tensor_slices(train_encodings)

# Training arguments
training_args = TrainingArguments(
    output_dir='./results', num_train_epochs=3, per_device_train_batch_size=2, 
    logging_dir='./logs', logging_steps=10, save_steps=10
)

# Define the Trainer for training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

# Train the model
trainer.train()

# Test the model on new sentence
test_sentence = ["Alice", "is", "a", "Data", "Scientist"]
test_encodings = tokenizer(test_sentence, return_tensors="pt", padding=True, truncation=True)
test_encodings.to(device)

# Predict with the trained model
outputs = model(**test_encodings)
logits = outputs.logits
predictions = torch.argmax(logits, dim=2)

# Decode and print test results
predicted_tags = [list(IBO_tags.keys())[pred] for pred in predictions[0].cpu().numpy()]
for word, tag in zip(test_sentence, predicted_tags[1:len(test_sentence)+1]):
    print(f"{word}: {tag}")


C:\Users\RAYMOND\AppData\Local\pypoetry\Cache\virtualenvs\text2uml-yWP-Bm0a-py3.9\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are in

NotImplementedError: return_offset_mapping is not available when using Python tokenizers. To use this feature, change your tokenizer to one deriving from transformers.PreTrainedTokenizerFast. More information on available tokenizers at https://github.com/huggingface/transformers/pull/2674

In [24]:
import pandas as pd

# Read the TSV file
df = pd.read_csv('./data/validation-set-sentence-labelled.tsv', sep='\t')

# Replace NaN values with False
df = df.fillna(False)

# Concatenate the Word, group by sentence_id, and sort by token_ID_within_sentence
grouped = df.groupby('sentence_ID').apply(lambda x: x.sort_values('token_ID_within_sentence')).reset_index(drop=True)

# Create a list of sentences with their corresponding labels
sentences = []
structure_focus = []
usecase_focus = []
process_focus = []

for name, group in grouped.groupby('sentence_ID'):
    sentence = ' '.join(group['word'].tolist())
    labels = group[['structure_focus', 'usecase_focus', 'process_focus']].iloc[0].tolist()
    sentences.append(sentence)
    structure_focus.append(labels[0])
    usecase_focus.append(labels[1])
    process_focus.append(labels[2])

# Create a DataFrame with sentences and their labels
result = pd.DataFrame({
    'sentence': sentences,
    'structure_focus': structure_focus,
    'usecase_focus': usecase_focus,
    'process_focus': process_focus
})

# Add a column for sentence length
result['sentence_length'] = result['sentence'].apply(len)

# Sort the DataFrame by sentence length
result = result.sort_values(by='sentence_length')

result

,sentence,structure_focus,usecase_focus,process_focus,sentence_length
39,"Yes , we need to know which documents were use...",True,False,False,59
38,AP Each : food item is identified by a unique ...,True,False,False,64
33,Analyst Some of : the Please tell menu me item...,False,False,False,104
37,Analyst Food : items Is are there utilized any...,False,False,False,114
32,"And For example I , must some record the of th...",True,True,True,145
30,AP Each Of menu : course for item the is class...,False,False,False,169
27,"So Yes There , , are what we many are do excit...",True,False,False,178
26,Analyst The We : menu only at allow Romano a '...,True,False,False,182
24,Not We When only like the that to reservation ...,True,False,False,209
31,"Absolutely The No . price , of we do each n’t ...",False,False,False,215
